# Практика 06. Деревья решений и ансамбли

**Версия:** 2026-09-24 (7b54111)

**Как сдавать работу**

1. Откройте ноутбук в Colab и сохраните копию себе: «Файл → Сохранить копию на Диске». Работайте в копии.
2. Выполните задания: код пишите вместо `# ВАШ КОД ЗДЕСЬ` / `raise NotImplementedError`,
   ответы на вопросы — после «*Ваш ответ:*».
3. После каждого задания запускайте ячейку с проверками. Проверки — для самоконтроля:
   их прохождение не гарантирует зачёт, а текстовые ответы проверяются отдельно.
4. Перед сдачей выполните «Среда выполнения → Перезапустить сеанс и выполнить все»: ноутбук должен
   выполниться целиком без ошибок.
5. Откройте доступ по ссылке («Настройки доступа → Все, у кого есть ссылка») и вставьте ссылку
   на свою копию в таблицу курса.

Свёрнутые ячейки со значком ▶ — служебные (загрузка данных, функции проверки). Их нужно выполнять,
но менять не нужно.

К лекции 06. План работы:

1. **Часть 1** — дерево решений своими руками: индекс Джини, поиск лучшего разбиения, построение
   дерева и предсказание; градиентный бустинг для регрессии из деревьев sklearn. Каждая функция
   сверяется с scikit-learn.
2. **Часть 2** — дерево, случайный лес и градиентный бустинг из scikit-learn на данных переписи США
   (Adult): подбор глубины дерева, сравнение моделей на тестовой выборке, важность признаков.
3. **Часть 3** — эксперимент и выводы: как качество зависит от глубины дерева, числа деревьев в лесу
   и шага бустинга, и почему. Код здесь простой, оценивается объяснение.

In [ ]:
# @title Служебная ячейка: импорты и функции проверки { display-mode: "form" }
import inspect
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0
rng = np.random.default_rng(SEED)


def assert_no_loops(func):
    """Проверяет, что в теле функции нет циклов for/while (включения списков тоже считаются)."""
    source = inspect.getsource(func)
    body = source.split('"""')[-1] if '"""' in source else source
    assert not re.search(r"\b(for|while)\b", body), (
        f"В функции {func.__name__} есть цикл. Здесь нужно решение без циклов — операциями над массивами."
    )


def make_tree_data(n=300, seed=1):
    """Синтетические данные для проверки дерева: 4 признака, 3 класса.

    Подобраны так, что ни в одном узле дерева глубины до 4 нет двух одинаково хороших разбиений,
    поэтому правильное дерево обязано совпасть с деревом sklearn.
    """
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, 4)) * [1, 2, 0.5, 3]
    y = ((X[:, 0] + 0.3 * X[:, 1] ** 2 + 0.5 * r.normal(size=n)) > 0.8).astype(int) + (X[:, 3] > 2)
    return X, y


print("Готово")

# Часть 1. Дерево решений и бустинг своими руками

Дерево для классификации с $K$ классами (метки $0, \dots, K - 1$) строится жадно: в каждом узле
перебираются все признаки $j$ и пороги $t$, и выбирается разбиение $Q \to Q_L, Q_R$ с наименьшей
средневзвешенной неопределённостью дочерних узлов
$$
\frac{|Q_L|}{|Q|} \operatorname{Gini}(Q_L) + \frac{|Q_R|}{|Q|} \operatorname{Gini}(Q_R), \qquad
\operatorname{Gini}(Q) = 1 - \sum_{k=0}^{K-1} p_k^2,
$$
где $p_k$ — доля объектов класса $k$ в узле. Это то же самое, что максимизировать прирост информации.

## Задание 1.1. Индекс Джини

Напишите `gini(y, K)` — индекс Джини для вектора меток `y` из $\{0, \dots, K-1\}$. Без циклов:
число объектов каждого класса даёт `np.bincount(y, minlength=K)`.

In [ ]:
def gini(y, K):
    """Индекс Джини выборки с метками y (целые числа от 0 до K-1)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
assert np.isclose(gini(np.array([0, 0, 0]), 2), 0), "В чистом узле индекс Джини равен 0"
assert np.isclose(gini(np.array([0, 1, 0, 1]), 2), 0.5), "Для двух классов поровну индекс Джини равен 0.5"
assert np.isclose(gini(np.array([0, 1, 2]), 3), 2 / 3), "Для трёх классов поровну индекс Джини равен 2/3"
assert np.isclose(gini(np.array([1, 1, 1, 0]), 3), 0.375), "Для долей (1/4, 3/4, 0) индекс Джини равен 0.375"
assert_no_loops(gini)
print("OK")

## Задание 1.2. Лучшее разбиение по одному признаку

Для одного признака `x` нужно перебрать все пороги. Различных разбиений не больше $n - 1$: имеет значение
только, какие объекты окажутся слева. Поэтому:

1. отсортируйте объекты по `x` (`np.argsort`);
2. для каждой позиции $i = 1, \dots, n - 1$ «слева» оказываются первые $i$ объектов отсортированного
   списка. Число объектов каждого класса слева для всех $i$ сразу — это накопленная сумма one-hot меток:
   `np.cumsum(np.eye(K)[y_sorted], axis=0)`. Справа — всё остальное;
3. по этим числам посчитайте для всех позиций сразу средневзвешенный индекс Джини потомков;
4. разбиение между равными значениями признака невозможно — такие позиции исключите
   (например, присвойте им `np.inf`);
5. верните порог посередине между соседними значениями лучшей позиции и её неопределённость.
   Если все значения признака одинаковы, верните `(None, np.inf)`.

Среди нескольких одинаково хороших позиций выбирайте первую (так ведёт себя `np.argmin`). Без циклов.

In [ ]:
def best_split_feature(x, y, K):
    """Лучший порог по признаку x: (threshold, weighted_gini) или (None, np.inf)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.tree import DecisionTreeClassifier


def _brute_force_split(x, y, K):
    values = np.unique(x)
    best = (None, np.inf)
    for a, b in zip(values[:-1], values[1:]):
        t = (a + b) / 2
        left, right = y[x <= t], y[x > t]
        imp = (len(left) * gini(left, K) + len(right) * gini(right, K)) / len(y)
        if imp < best[1] - 1e-12:
            best = (t, imp)
    return best


X_chk, y_chk = make_tree_data()
for j in range(4):
    t, imp = best_split_feature(X_chk[:, j], y_chk, 3)
    t_ref, imp_ref = _brute_force_split(X_chk[:, j], y_chk, 3)
    assert t is not None and np.isclose(t, t_ref), f"Признак {j}: порог {t}, а должен быть {t_ref}"
    assert np.isclose(imp, imp_ref), f"Признак {j}: неопределённость {imp:.4f}, а должна быть {imp_ref:.4f}"
    stump = DecisionTreeClassifier(max_depth=1).fit(X_chk[:, [j]], y_chk)
    assert np.isclose(t, stump.tree_.threshold[0]), f"Признак {j}: порог не совпал со sklearn"
assert best_split_feature(np.ones(5), np.array([0, 1, 0, 1, 0]), 2) == (None, np.inf), (
    "Если все значения признака одинаковы, разбиения нет: нужно вернуть (None, np.inf)"
)
t, _ = best_split_feature(np.array([1.0, 1.0, 2.0, 3.0]), np.array([0, 1, 1, 1]), 2)
assert t is not None and t > 1.0, "Нельзя ставить порог между равными значениями признака"
assert_no_loops(best_split_feature)
print("OK")

## Задание 1.3. Лучшее разбиение по всем признакам

Напишите `best_split(X, y, K)`: переберите признаки (здесь цикл по признакам уместен — их немного),
для каждого найдите лучший порог функцией `best_split_feature` и верните тройку
`(j, threshold, weighted_gini)` для лучшего признака. При равенстве выбирайте признак с меньшим номером.
Если разбить нельзя ни по одному признаку, верните `(None, None, np.inf)`.

In [ ]:
def best_split(X, y, K):
    """(j, threshold, weighted_gini) лучшего разбиения или (None, None, np.inf)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
j, t, imp = best_split(X_chk, y_chk, 3)
root = DecisionTreeClassifier(max_depth=1).fit(X_chk, y_chk).tree_
assert j == root.feature[0], f"Лучший признак в корне — {root.feature[0]}, а получен {j}"
assert np.isclose(t, root.threshold[0]), f"Порог в корне — {root.threshold[0]:.4f}, а получен {t}"
assert best_split(np.ones((4, 2)), np.array([0, 1, 0, 1]), 2) == (None, None, np.inf), (
    "Если разбить нельзя ни по одному признаку, нужно вернуть (None, None, np.inf)"
)
print("OK")

## Задание 1.4. Построение дерева и предсказание

Дерево будем хранить как вложенные словари:

- лист: `{"leaf": класс}` — самый частый класс среди объектов узла (при равенстве — меньший номер,
  как у `np.argmax` от `np.bincount`);
- внутренний узел: `{"feature": j, "threshold": t, "left": поддерево, "right": поддерево}`;
  объекты с `x[j] <= t` идут налево.

Напишите рекурсивную функцию `build_tree(X, y, K, max_depth, depth=0)`. Узел становится листом, если
достигнута максимальная глубина, если он чистый (все объекты одного класса) или если разбить его нельзя.
Затем напишите `predict_tree(tree, X)` — предсказания для всех строк `X` (здесь цикл по объектам допустим).

In [ ]:
def build_tree(X, y, K, max_depth, depth=0):
    """Дерево в виде вложенных словарей."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def predict_tree(tree, X):
    """Вектор предсказанных классов для строк X."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
X_new = np.random.default_rng(7).normal(size=(500, 4)) * [1, 2, 0.5, 3]
for depth in [1, 2, 3, 4]:
    tree = build_tree(X_chk, y_chk, 3, max_depth=depth)
    sk = DecisionTreeClassifier(max_depth=depth).fit(X_chk, y_chk)
    ours = predict_tree(tree, X_new)
    assert ours.shape == (500,), f"predict_tree должна вернуть вектор длины 500, получено {np.shape(ours)}"
    assert np.array_equal(ours, sk.predict(X_new)), (
        f"Глубина {depth}: предсказания расходятся со sklearn в {np.sum(ours != sk.predict(X_new))} объектах"
    )
assert build_tree(X_chk, y_chk, 3, max_depth=0) == {"leaf": int(np.bincount(y_chk).argmax())}, (
    "При max_depth=0 дерево — один лист с самым частым классом"
)
assert build_tree(X_chk[y_chk == 1], y_chk[y_chk == 1], 3, max_depth=5) == {"leaf": 1}, "Чистый узел не должен разбиваться"
print("OK")

## Задание 1.5. Градиентный бустинг для регрессии

С квадратичной функцией потерь $\ell(y, F) = \frac{1}{2}(y - F)^2$ псевдо-остатки градиентного
бустинга — обычные остатки $r_i = y_i - F_{t-1}(\mathbf{x}_i)$, а начальное приближение — среднее $\bar{y}$.
Алгоритм:

1. $F_0 = \bar{y}$;
2. для $t = 1, \dots, T$: обучить `DecisionTreeRegressor(max_depth=max_depth, random_state=SEED)` на
   парах $(\mathbf{x}_i, r_i)$ и обновить $F_t(\mathbf{x}) = F_{t-1}(\mathbf{x}) + \eta\, h_t(\mathbf{x})$.

Напишите `fit_gb(X, y, n_trees, learning_rate, max_depth)`, возвращающую `(F0, trees)`, и
`predict_gb(F0, trees, X, learning_rate)`. Результат должен совпасть с `GradientBoostingRegressor`.

In [ ]:
from sklearn.tree import DecisionTreeRegressor


def fit_gb(X, y, n_trees, learning_rate, max_depth):
    """Возвращает (F0, список деревьев)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def predict_gb(F0, trees, X, learning_rate):
    """Предсказание композиции F0 + learning_rate * сумма деревьев."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

r = np.random.default_rng(1)
X_reg = r.normal(size=(300, 3))
y_reg = np.sin(X_reg[:, 0]) + X_reg[:, 1] ** 2 + 0.3 * r.normal(size=300)
X_reg_new = r.normal(size=(100, 3))

# Не больше 15 деревьев: дальше в мелких узлах появляются равноценные разбиения,
# и sklearn выбирает между ними случайно — предсказания на новых точках могут разойтись.
F0, trees = fit_gb(X_reg, y_reg, n_trees=15, learning_rate=0.1, max_depth=3)
assert np.isclose(F0, y_reg.mean()), "F0 должно быть средним y"
assert len(trees) == 15, f"Должно быть 15 деревьев, получено {len(trees)}"
sk = GradientBoostingRegressor(
    n_estimators=15, learning_rate=0.1, max_depth=3, criterion="squared_error", random_state=SEED
).fit(X_reg, y_reg)
ours = predict_gb(F0, trees, X_reg_new, 0.1)
assert np.allclose(ours, sk.predict(X_reg_new)), (
    f"Предсказания расходятся с GradientBoostingRegressor (макс. разница {np.abs(ours - sk.predict(X_reg_new)).max():.4f}). "
    "Каждое дерево обучается на остатках y - F текущей модели"
)
print("OK")

# Часть 2. Деревья и ансамбли в scikit-learn: данные Adult

Данные переписи населения США 1994 года (UCI Adult, 48 842 человека): возраст, образование, семейное
положение, профессия, число рабочих часов в неделю, доходы и убытки от капитала и т. д. Нужно
предсказать, превышает ли годовой доход 50 тысяч долларов (класс 1, около 24% объектов).
Признак `fnlwgt` — вес записи в выборке переписи (сколько человек в населении она представляет);
признак `education` удалён, так как дублирует `education-num`.

In [ ]:
# @title Загрузка данных: Adult (OpenML, id 1590) и разбиение { display-mode: "form" }
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

adult = fetch_openml(data_id=1590, as_frame=True, parser="auto").frame
y_all = (adult["class"] == ">50K").astype(int).to_numpy()
X_all = adult.drop(columns=["class", "education"])
CATEGORICAL = [c for c in X_all.columns if str(X_all[c].dtype) == "category"]
NUMERIC = [c for c in X_all.columns if c not in CATEGORICAL]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all
)
print(f"обучение: {X_train.shape}, тест: {X_test.shape}, доля класса 1: {y_all.mean():.3f}")
print("категориальные:", CATEGORICAL)
print("числовые:", NUMERIC)
X_train.head()

## Задание 2.1. Предобработка

Напишите `make_preprocessor()`, возвращающую `ColumnTransformer`, который:

- кодирует категориальные признаки `CATEGORICAL` one-hot: `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`
  (плотная матрица нужна градиентному бустингу; пропуски энкодер считает отдельной категорией);
- пропускает числовые признаки без изменений (`remainder="passthrough"`).

Масштабировать числовые признаки не нужно — почему, вы объясните в части 3.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


def make_preprocessor():
    """ColumnTransformer: one-hot для категориальных признаков, числовые — как есть."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
prep = make_preprocessor()
assert isinstance(prep, ColumnTransformer), "make_preprocessor должна возвращать ColumnTransformer"
Z = prep.fit_transform(X_train)
assert isinstance(Z, np.ndarray), "Результат должен быть плотной матрицей: укажите sparse_output=False"
assert not np.isnan(Z).any(), "После предобработки остались пропуски: они должны стать отдельной категорией one-hot"
assert Z.shape[0] == len(X_train) and Z.shape[1] > 50, f"Ожидалось больше 50 столбцов после one-hot, получено {Z.shape[1]}"
assert "StandardScaler" not in repr(prep), "Масштабирование деревьям не нужно"
assert np.array_equal(Z[:, -len(NUMERIC):], X_train[NUMERIC].to_numpy()), "Числовые признаки должны пройти без изменений (remainder='passthrough')"
print("OK")

## Задание 2.2. Дерево решений: подбор глубины

Подберите глубину дерева поиском по сетке. Конвейер: `make_preprocessor()` и
`DecisionTreeClassifier(random_state=SEED)` с именем шага `"tree"`. Сетка — `DEPTHS`, метрика —
`scoring="roc_auc"`, кросс-валидация — `CV`. Укажите `return_train_score=True`: качество на обучающих
фолдах понадобится в части 3. Результат — `grid_tree`, обученный на `X_train`.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

DEPTHS = [2, 4, 6, 8, 10, 12, 16, 20, None]
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("лучшая глубина:", grid_tree.best_params_, f"ROC AUC (CV) = {grid_tree.best_score_:.3f}")

In [ ]:
assert isinstance(grid_tree, GridSearchCV), "grid_tree должен быть GridSearchCV"
assert grid_tree.scoring == "roc_auc", "Метрика должна быть roc_auc"
assert "mean_train_score" in grid_tree.cv_results_, "Нужен return_train_score=True"
assert len(grid_tree.cv_results_["params"]) == len(DEPTHS), "В сетке должны быть все значения DEPTHS"
assert grid_tree.best_params_["tree__max_depth"] is not None, "Дерево без ограничения глубины не должно оказаться лучшим — проверьте сетку и метрику"
assert grid_tree.best_score_ > 0.85, f"ROC AUC на кросс-валидации подозрительно низкий: {grid_tree.best_score_:.3f}"
print("OK")

## Задание 2.3. Случайный лес и градиентный бустинг

Обучите на `X_train` два конвейера с `make_preprocessor()`:

- `forest` — `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED)`;
- `boosting` — `HistGradientBoostingClassifier(max_iter=500, learning_rate=0.1, early_stopping=True, random_state=SEED)`.
  Это быстрая реализация градиентного бустинга на гистограммах признаков (как в XGBoost и LightGBM).
  С `early_stopping=True` она откладывает 10% обучающих данных и останавливается, когда качество на них
  перестаёт расти.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("деревьев в бустинге после ранней остановки:", boosting[-1].n_iter_)

In [ ]:
assert isinstance(forest[-1], RandomForestClassifier) and forest[-1].n_estimators == 200, "forest: нужен RandomForestClassifier со 200 деревьями"
assert forest[-1].min_samples_leaf == 5, "forest: min_samples_leaf=5"
assert isinstance(boosting[-1], HistGradientBoostingClassifier), "boosting: нужен HistGradientBoostingClassifier"
assert boosting[-1].n_iter_ < 500, "boosting: ранняя остановка должна сработать раньше 500 итераций (early_stopping=True)"
print("OK")

## Задание 2.4. Сравнение на тестовой выборке

Заполните словарь `results`: для каждой модели — `"tree"` (лучшее дерево `grid_tree.best_estimator_`),
`"forest"`, `"boosting"` — словарь с ROC AUC (по `predict_proba(X_test)[:, 1]`) и accuracy на тестовой выборке.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

models = {"tree": grid_tree.best_estimator_, "forest": forest, "boosting": boosting}
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

pd.DataFrame(results).T.round(4)

In [ ]:
assert set(results) == {"tree", "forest", "boosting"}, f"Нужны ключи tree, forest, boosting; получено {set(results)}"
for name, model in models.items():
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    assert np.isclose(results[name]["roc_auc"], auc), f"{name}: ROC AUC посчитан неверно: нужны вероятности predict_proba(X_test)[:, 1] на тестовой выборке"
    assert np.isclose(results[name]["accuracy"], accuracy_score(y_test, model.predict(X_test))), f"{name}: accuracy посчитана неверно: нужны предсказания predict(X_test) на тестовой выборке"
assert results["boosting"]["roc_auc"] > results["tree"]["roc_auc"], "Бустинг должен быть лучше одного дерева — проверьте модели"
print("OK")

## Задание 2.5. Важность признаков

Сравним два способа оценить важность признаков.

- **MDI** случайного леса уже посчитан при обучении: `forest[-1].feature_importances_`. Он относится
  к столбцам после one-hot, их имена — `forest[0].get_feature_names_out()`.
- **Перестановочная важность** (`sklearn.inspection.permutation_importance`) считается для исходных
  признаков: значения признака перемешиваются и измеряется падение качества.

Посчитайте:

- `mdi` — `pd.Series` важностей MDI леса с индексом из имён столбцов после one-hot, по убыванию;
- `perm` — `pd.Series` перестановочной важности **леса** `forest` на первых 3000 объектах тестовой
  выборки (`n_repeats=5`, `random_state=SEED`, `scoring="roc_auc"`), индекс — имена исходных признаков
  (`X_test.columns`), по убыванию. Возьмите `importances_mean`.

In [ ]:
from sklearn.inspection import permutation_importance

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mdi.head(10)[::-1].plot.barh(ax=axes[0], title="MDI (лес), топ-10 столбцов")
perm[::-1].plot.barh(ax=axes[1], title="Перестановочная важность (лес), падение ROC AUC")
plt.tight_layout()
plt.show()

In [ ]:
assert isinstance(mdi, pd.Series) and len(mdi) == len(forest[0].get_feature_names_out()), "mdi: по одному значению на каждый столбец после one-hot"
assert np.isclose(mdi.sum(), 1), "Важности MDI в sklearn нормированы: сумма должна быть 1"
assert mdi.is_monotonic_decreasing and perm.is_monotonic_decreasing, "mdi и perm должны быть отсортированы по убыванию"
assert set(perm.index) == set(X_test.columns), "perm: индекс — имена исходных признаков X_test.columns"
print("OK")

# Часть 3. Эксперимент и выводы

## Задание 3.1. Глубина дерева

По `grid_tree.cv_results_` постройте график ROC AUC на обучающих фолдах (`mean_train_score`) и на
кросс-валидации (`mean_test_score`) в зависимости от глубины. Дерево без ограничения (`None`) подпишите
на оси как «∞». Сохраните значения в массивы `train_auc` и `cv_auc` в порядке `DEPTHS`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert len(train_auc) == len(DEPTHS) and len(cv_auc) == len(DEPTHS), "train_auc и cv_auc — по значению на каждую глубину"
assert np.isclose(train_auc[-1], 1.0), "У дерева без ограничения глубины ROC AUC на обучении должен быть 1"
assert cv_auc.argmax() < len(DEPTHS) - 1, "Максимум на кросс-валидации не должен приходиться на дерево без ограничения"
print("OK")

## Задание 3.2. Число деревьев в лесу

Для $B \in$ `N_TREES` обучите случайный лес (параметры как в задании 2.3, кроме `n_estimators`) и
посчитайте ROC AUC на тестовой выборке. Сохраните результаты в список `forest_auc` и постройте график
зависимости от $B$ (удобна логарифмическая шкала по $B$: `plt.xscale("log")`).

Здесь мы используем тест только для иллюстрации: гиперпараметр по этому графику не выбирается.

In [ ]:
N_TREES = [1, 3, 10, 30, 100, 300]
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert len(forest_auc) == len(N_TREES), "forest_auc — по значению на каждое B"
assert forest_auc[-1] > forest_auc[0] + 0.01, "Лес из 300 деревьев должен быть заметно лучше одного дерева"
print("OK")

## Задание 3.3. Шаг бустинга

Для `learning_rate` из `LEARNING_RATES` обучите `HistGradientBoostingClassifier(max_iter=300,
learning_rate=..., early_stopping=False, random_state=SEED)` в конвейере с `make_preprocessor()`.
Метод `staged_predict_proba` возвращает предсказания после каждой итерации. Посчитайте по ним ROC AUC
на тестовой выборке после каждой итерации и сохраните в словарь `gb_curves`: шаг → список из 300 значений.
Постройте обе кривые на одном графике.

Модель в конвейере: сначала преобразуйте `X_test` предобработкой (`model[0].transform`), затем вызовите
`model[-1].staged_predict_proba`.

In [ ]:
LEARNING_RATES = [0.5, 0.05]
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert set(gb_curves) == set(LEARNING_RATES), "В gb_curves нужны оба значения шага"
assert all(len(c) == 300 for c in gb_curves.values()), "Для каждого шага — 300 значений ROC AUC"
assert max(gb_curves[0.5]) > gb_curves[0.5][-1], "При большом шаге качество на тесте к концу должно снижаться"
print("OK")

## Задание 3.4. Выводы

Ответьте на вопросы, опираясь на свои графики и числа. Ответ на каждый вопрос — 2–4 предложения.

1. Почему мы не масштабировали числовые признаки? Изменилось бы дерево, если бы мы применили
   `StandardScaler`? А если бы прологарифмировали признак `capital-gain`?
2. Как ROC AUC на обучении и на кросс-валидации зависит от глубины дерева (задание 3.1)? Объясните
   в терминах смещения и разброса. Какая глубина оказалась лучшей?
3. Как качество леса зависит от числа деревьев (задание 3.2)? Почему лес не переобучается с ростом $B$
   и почему после некоторого $B$ качество почти перестаёт расти? Свяжите с формулой дисперсии среднего
   коррелированных моделей из лекции.
4. Как ведёт себя бустинг при разных шагах (задание 3.3)? Почему большой шаг хуже, а малому нужно больше
   деревьев? Зачем нужна ранняя остановка?
5. Сравните три модели по качеству на тесте (задание 2.4). Какая лучше и почему именно она, если
   вспомнить, что уменьшает бэггинг, а что — бустинг?
6. Сравните важность признаков по MDI и перестановочную (задание 2.5). Какие признаки важны по обеим
   мерам? Найдите признак `fnlwgt` в обоих рейтингах (`mdi["remainder__fnlwgt"]`, его место в `mdi`
   и в `perm`). Чем объясняется разница и почему MDI может завышать важность таких признаков?

*Ваш ответ:*